# DQN Pong group notebook

Alliane: environment and gameplay. Anthony: learning algorithm. HonourGod: repo, results and presentation.

This notebook tests our code. It doesn't hold any training code itself: everything is imported from `environment.py`, `network.py`, `replay_buffer.py` and `train.py`, so there is only one copy of each function.

How to use it:
1. Write your code in the `.py` file, not here. Anthony's and HonourGod's functions are in `train.py`.
2. Keep the function names and arguments the same as in `train.py`. The other parts call them.
3. Restart the kernel and run all cells. Each test prints `PASS`, or `NOT WRITTEN YET` if the function still raises `NotImplementedError`.
4. When every test passes, run the smoke test in section 6. Nobody starts their 10 experiments before that.

## 1. Setup (everyone)
Runs locally with the project's `.venv` kernel, or on Google Colab. On Colab it clones the repo and installs the Atari packages with uv. Change `BRANCH` to the branch you want to test.

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/H-levison/dqn-atari.git"
    BRANCH = "main"  # or your own branch
    if not os.path.exists("repo"):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, "repo"], check=True)
    os.chdir("repo")
    # Colab already has torch, numpy, pandas and matplotlib
    subprocess.run(["uv", "pip", "install", "--system", "gymnasium", "ale-py", "opencv-python"], check=True)

print("Working directory:", os.getcwd())

In [ ]:
import copy
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from environment import make_env
from network import DQN
from replay_buffer import ReplayBuffer
from train import (
    LOG_COLUMNS,
    build_networks,
    get_epsilon,
    learn,
    parse_args,
    select_action,
    start_log,
    train,
    update_target,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

N_ACTIONS = 6  # Pong has 6 actions


def run_test(test_function):
    # Runs a test and prints the result. A function that isn't written yet
    # raises NotImplementedError, which we report instead of crashing.
    try:
        test_function()
        print("PASS:", test_function.__name__)
    except NotImplementedError:
        print("NOT WRITTEN YET:", test_function.__name__)

## 2. Training settings (everyone)
The hyperparameters are command-line flags in `train.py`. Their defaults are the values we agreed on, and they are the same for every experiment unless you change them on purpose:

In [ ]:
# Same output as typing `uv run train.py --help` in a terminal
result = subprocess.run([sys.executable, "train.py", "--help"], capture_output=True, text=True)
print(result.stdout)

## 3. Environment, network and replay buffer (Alliane, done)
- `make_env(seed)` gives observations of shape `(4, 84, 84)` with dtype uint8.
- `DQN(n_actions)` takes a batch of states and returns Q-values of shape `(batch, n_actions)`. You can give it uint8 tensors directly.
- `ReplayBuffer(capacity)` has `add(s, a, r, s2, done)` and `sample(batch_size)`. `sample` returns numpy arrays in the order states, actions, rewards, next_states, dones. `len(buffer)` is the number of transitions stored.

In [ ]:
def test_environment_network_buffer():
    env = make_env(seed=0)
    state, _ = env.reset()
    assert state.shape == (4, 84, 84)
    assert state.dtype == np.uint8
    assert env.action_space.n == N_ACTIONS

    # Play 200 random steps and store them in the buffer
    buffer = ReplayBuffer(capacity=500)
    for _ in range(200):
        action = env.action_space.sample()
        next_state, reward, terminated, truncated, _ = env.step(action)
        buffer.add(state, action, reward, next_state, terminated)
        state = next_state
        if terminated or truncated:
            state, _ = env.reset()
    assert len(buffer) == 200

    # Sample a batch and pass it through the network
    states, actions, rewards, next_states, dones = buffer.sample(32)
    network = DQN(N_ACTIONS)
    q_values = network(torch.as_tensor(states))
    assert q_values.shape == (32, N_ACTIONS)
    env.close()


run_test(test_environment_network_buffer)

## 4. Epsilon-greedy, epsilon decay and CSV logging (HonourGod)
In `train.py`:
- `get_epsilon(step, eps_start, eps_end, eps_decay)`: epsilon goes down in a straight line from `eps_start` to `eps_end` over `eps_decay` steps, then stays at `eps_end`.
- `select_action(state, online_net, epsilon, n_actions, device)`: with probability `epsilon` a random action, otherwise the action with the highest Q-value.
- `start_log(out_dir)`: creates the run folder, opens `log.csv`, writes the header and returns `(file, writer)`. The training loop writes one row per episode.

In [ ]:
def test_get_epsilon():
    assert get_epsilon(0, 1.0, 0.02, 1000) == 1.0
    assert abs(get_epsilon(500, 1.0, 0.02, 1000) - 0.51) < 1e-6  # halfway down
    assert abs(get_epsilon(1000, 1.0, 0.02, 1000) - 0.02) < 1e-6  # reached eps_end
    assert abs(get_epsilon(5000, 1.0, 0.02, 1000) - 0.02) < 1e-6  # stays at eps_end


def test_select_action():
    network = DQN(N_ACTIONS).to(device)
    state = np.random.randint(0, 256, size=(4, 84, 84), dtype=np.uint8)

    # With epsilon = 0 the agent must always pick the best action
    with torch.no_grad():
        q_values = network(torch.as_tensor(state, device=device).unsqueeze(0))
    best_action = int(q_values.argmax())
    assert select_action(state, network, 0.0, N_ACTIONS, device) == best_action

    # With epsilon = 1 the agent must pick random actions
    actions = [select_action(state, network, 1.0, N_ACTIONS, device) for _ in range(200)]
    for action in actions:
        assert 0 <= action < N_ACTIONS
    assert len(set(actions)) > 1, "epsilon = 1 should give different actions"


def test_start_log():
    out_dir = Path("smoke_test/log_test")
    log_file, writer = start_log(out_dir)
    writer.writerow([1, 900, -21.0, 900, 0.99, 0.01])
    log_file.close()

    log = pd.read_csv(out_dir / "log.csv")
    assert list(log.columns) == LOG_COLUMNS
    assert len(log) == 1  # one episode written under the header


run_test(test_get_epsilon)
run_test(test_select_action)
run_test(test_start_log)

## 5. Networks, learning update and training loop (Anthony)
In `train.py`:
- `build_networks(n_actions, device)`: returns `(online_net, target_net)`, with the target starting as a copy of the online weights.
- `update_target(online_net, target_net)`: copies the online weights into the target network.
- `learn(online_net, target_net, buffer, optimizer, batch_size, gamma, device)`: one gradient step. Q(s, a) comes from the online network, and the Bellman target `r + gamma * max Q_target(s', a') * (1 - done)` comes from the target network with no gradients. Returns the loss as a float.
- `train(args)`: the main loop. It saves the model to `<out>/model.pth` and the log to `<out>/log.csv`.

In [ ]:
def test_build_networks():
    online_net, target_net = build_networks(N_ACTIONS, device)
    online_weights = online_net.state_dict()
    target_weights = target_net.state_dict()
    for name in online_weights:
        assert torch.equal(online_weights[name], target_weights[name])


def test_update_target():
    online_net = DQN(N_ACTIONS)
    target_net = DQN(N_ACTIONS)
    update_target(online_net, target_net)

    online_weights = online_net.state_dict()
    target_weights = target_net.state_dict()
    for name in online_weights:
        assert torch.equal(online_weights[name], target_weights[name])


def test_learn():
    online_net = DQN(N_ACTIONS).to(device)
    target_net = copy.deepcopy(online_net)
    optimizer = torch.optim.Adam(online_net.parameters(), lr=1e-4)

    # Fill a small buffer with random transitions
    buffer = ReplayBuffer(capacity=100)
    for _ in range(100):
        state = np.random.randint(0, 256, size=(4, 84, 84), dtype=np.uint8)
        next_state = np.random.randint(0, 256, size=(4, 84, 84), dtype=np.uint8)
        action = random.randrange(N_ACTIONS)
        reward = random.choice([-1.0, 0.0, 1.0])
        buffer.add(state, action, reward, next_state, False)

    online_before = copy.deepcopy(online_net.state_dict())
    target_before = copy.deepcopy(target_net.state_dict())

    loss = learn(online_net, target_net, buffer, optimizer, 32, 0.99, device)
    assert isinstance(loss, float)

    # The online network should change, the target network should not
    online_changed = False
    for name, weights in online_net.state_dict().items():
        if not torch.equal(weights, online_before[name]):
            online_changed = True
    assert online_changed, "learn() did not update the online network"

    for name, weights in target_net.state_dict().items():
        assert torch.equal(weights, target_before[name]), "learn() should not change the target network"


run_test(test_build_networks)
run_test(test_update_target)
run_test(test_learn)

## 6. Integration smoke test (everyone)
Runs `train()` for 5,000 steps. It passes if `model.pth` and `log.csv` are saved, the log has the agreed columns, the loss is never `nan`, and the saved model loads the same way `play.py` loads it.

The reward won't improve in 5,000 steps. This run only checks that the parts work together.

In [ ]:
# parse_args() reads sys.argv, so we set it to the flags we would type in a terminal.
# Every other hyperparameter keeps its default from train.py.
sys.argv = ["train.py", "--steps", "5000", "--warmup", "1000", "--buffer-size", "10000", "--out", "smoke_test/run"]
smoke_args = parse_args()


def test_integration():
    start = time.time()
    train(smoke_args)
    print(f"Training took {time.time() - start:.0f} seconds")

    model_path = Path(smoke_args.out) / "model.pth"
    csv_path = Path(smoke_args.out) / "log.csv"
    assert model_path.exists(), "no model.pth was saved"
    assert csv_path.exists(), "no log.csv was written"

    log = pd.read_csv(csv_path)
    print(log)
    assert list(log.columns) == LOG_COLUMNS
    assert not log["loss"].isna().any(), "the loss has nan values"

    # Load the weights the same way play.py does
    network = DQN(N_ACTIONS)
    network.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))


run_test(test_integration)

## 7. Training plots (Alliane)
Set `csv_path` to the log of the run you want to plot, for example `experiments/Alliane/exp03/log.csv`.

In [ ]:
from plot_results import plot_training

csv_path = Path("smoke_test/run/log.csv")
if csv_path.exists():
    plot_training(pd.read_csv(csv_path), title=csv_path.parent.name)
    plt.show()
else:
    print(csv_path, "doesn't exist yet. Run the smoke test or an experiment first.")

## 8. Watching the agent play (Alliane)
The game window can't open in a notebook or on Colab, so run `play.py` from a terminal on your laptop:

```
uv run play.py                                           # plays dqn_model.pth
uv run play.py --model experiments/Alliane/exp03/model.pth
uv run play.py --episodes 5
```